# 🤟 HST-GNN v2 — WLASL Sign Language Recognition
**Dataset:** WLASL (Isolated ASL, ~2000 classes)  
**GPU target:** T4 15GB (Colab free)  
**Pipeline:** MediaPipe keypoints → Graph Encoder → CE Classification  

### 📋 Thứ tự chạy:
1. Cell 1: Mount Drive  
2. Cell 2: Check GPU  
3. Cell 3: Install (→ **Restart Runtime** → tiếp từ Cell 4)  
4. Cell 4: Upload code files  
5. Cell 5: Kaggle setup  
6. Cell 6: Download WLASL  
7. Cell 7: Extract keypoints (chậm ~30-60 phút)  
8. Cell 8: Build vocab + Config  
9. Cell 9: Sanity check  
10. Cell 10: Train  
11. Cell 11: Visualize  
12. Cell 12: Evaluate  


## 1️⃣ Mount Google Drive

In [1]:
from google.colab import drive
import os
from google.colab._message import MessageError # Import MessageError for specific error handling

try:
    drive.mount('/content/drive')

    DRIVE_ROOT = '/content/drive/MyDrive/sign_language_wlasl'
    DRIVE_CKPT = f'{DRIVE_ROOT}/checkpoints'
    DRIVE_DATA = f'{DRIVE_ROOT}/data'
    os.makedirs(DRIVE_CKPT, exist_ok=True)
    os.makedirs(DRIVE_DATA, exist_ok=True)
    print(f'✓ Drive mounted')
    print(f'  Checkpoint dir : {DRIVE_CKPT}')
    print(f'  Data dir       : {DRIVE_DATA}')
except MessageError as e:
    print(f'❌ Error mounting Google Drive: {e}')
    print('💡 Please try running this cell again. If the issue persists, restart the Colab runtime (Runtime > Restart runtime) and then re-run this cell.')
except Exception as e:
    print(f'❌ An unexpected error occurred: {e}')

Mounted at /content/drive
✓ Drive mounted
  Checkpoint dir : /content/drive/MyDrive/sign_language_wlasl/checkpoints
  Data dir       : /content/drive/MyDrive/sign_language_wlasl/data


## 2️⃣ Check GPU

In [2]:
import torch
print(f'GPU   : {torch.cuda.get_device_name(0)}')
print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb < 14:
    BATCH_SIZE = 2; D_MODEL = 128
    print('⚠️  VRAM < 14GB → batch=2, d_model=128')
elif vram_gb < 20:
    BATCH_SIZE = 4; D_MODEL = 256
    print('✓ T4/P100 → batch=4, d_model=256')
else:
    BATCH_SIZE = 8; D_MODEL = 256
    print('🚀 A100 → batch=8, d_model=256')
print(f'→ batch_size={BATCH_SIZE}, d_model={D_MODEL}')


GPU   : Tesla T4
VRAM  : 15.6 GB
PyTorch: 2.10.0+cu128
✓ T4/P100 → batch=4, d_model=256
→ batch_size=4, d_model=256


## 3️⃣ Install Dependencies
> ⚠️ **SAU KHI CELL NÀY CHẠY XONG → Runtime → Restart runtime → tiếp từ Cell 4**


In [3]:
# Cài đúng thứ tự: numpy trước, rồi mới mediapipe
# Không dùng %%capture để dễ debug nếu lỗi

# 1. Gỡ numpy cũ, cài version tương thích
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'WARN: {cmd}\n{r.stderr[-300:]}')
    return r.returncode

print('Step 1/5: Pin numpy 1.26.4 ...')
run('pip install "numpy==1.26.4" -q --force-reinstall')

print('Step 2/5: Install mediapipe 0.10.14 (compatible với numpy 1.x) ...')
run('pip uninstall mediapipe -y -q')
run('pip install "mediapipe==0.10.14" -q')

print('Step 3/5: Install opencv ...')
run('pip install opencv-python-headless -q')

print('Step 4/5: Install torch tools ...')
run('pip install "transformers>=4.40.0,<4.46.0" accelerate -q')

print('Step 5/5: Install utils ...')
run('pip install kagglehub pyyaml tqdm einops jiwer sacrebleu -q')

print()
print('=' * 50)
print('✅ DONE — Bây giờ: Runtime → Restart runtime')
print('   Sau đó tiếp tục từ Cell 4 (KHÔNG chạy lại Cell này)')
print('=' * 50)


Step 1/5: Pin numpy 1.26.4 ...
Step 2/5: Install mediapipe 0.10.14 (compatible với numpy 1.x) ...
Step 3/5: Install opencv ...
Step 4/5: Install torch tools ...
Step 5/5: Install utils ...

✅ DONE — Bây giờ: Runtime → Restart runtime
   Sau đó tiếp tục từ Cell 4 (KHÔNG chạy lại Cell này)


In [ ]:
import subprocess, sys

# Gỡ numpy cũ hoàn toàn
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'numpy', '-y'], capture_output=True)

# Cài numpy 1.26.4
r = subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy==1.26.4', '--force-reinstall'],
                   capture_output=True, text=True)
print(r.stdout[-300:])
print(r.stderr[-200:] if r.stderr else '')

# Verify pip thấy đúng version
r2 = subprocess.run([sys.executable, '-m', 'pip', 'show', 'numpy'], capture_output=True, text=True)
print(r2.stdout)
print('✅ Nếu thấy Version: 1.26.4 ở trên → Restart Runtime ngay!')

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)

py 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.

Name: numpy
Version: 1.26.4
Summary: Fundamental package for array computing in Python
Home-page: https://numpy.org
Author: Travis E. Oliphant et al.
Author-email: 
License: Copyright (c) 2005-2023, NumPy Developers.
All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are
met:

    * Redistributions of source code must retain the above copyright
       notice, this list of conditions and the following disclaimer.

    * Redistributions in binary form must reproduce the above
       copyrigh

In [5]:
!pip install numpy==1.26.4

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.4
    Uninstalling numpy-2.4.4:
      Successfully uninstalled numpy-2.4.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2

## 4️⃣ Upload Code (chạy sau khi Restart Runtime)

In [1]:
# Verify numpy version trước
import numpy as np
print(f'numpy: {np.__version__}')
assert np.__version__.startswith('1.'), f'❌ numpy sai version: {np.__version__} — chạy lại Cell 3 rồi Restart Runtime!'
print('✓ numpy version OK')

# Verify mediapipe
import mediapipe as mp
print(f'mediapipe: {mp.__version__}')
print('✓ mediapipe import OK')

# Option A: Clone từ GitHub (nếu đã push code)
# !git clone https://github.com/quocvietpham185/sign_language_translation /content/sign_language
# %cd /content/sign_language

# Option B: Upload thủ công
import os
os.makedirs('/content/sign_language', exist_ok=True)

from google.colab import files
print('\nUpload 7 files: train.py, model.py, dataset.py, config.py, utils.py, vocabulary.py, scheduler.py')
uploaded = files.upload()
for fname in uploaded:
    dest = f'/content/sign_language/{fname}'
    os.rename(fname, dest)
    print(f'  ✓ {fname}')

%cd /content/sign_language
import subprocess
r = subprocess.run(['ls', '-la'], capture_output=True, text=True)
print(r.stdout)


numpy: 1.26.4
✓ numpy version OK
mediapipe: 0.10.14
✓ mediapipe import OK

Upload 7 files: train.py, model.py, dataset.py, config.py, utils.py, vocabulary.py, scheduler.py


Saving config.py to config.py
Saving dataset.py to dataset.py
Saving model.py to model.py
Saving scheduler.py to scheduler.py
Saving train.py to train.py
Saving utils.py to utils.py
Saving vocabulary.py to vocabulary.py
  ✓ config.py
  ✓ dataset.py
  ✓ model.py
  ✓ scheduler.py
  ✓ train.py
  ✓ utils.py
  ✓ vocabulary.py
/content/sign_language
total 116
drwxr-xr-x 2 root root  4096 Apr  7 03:15 .
drwxr-xr-x 1 root root  4096 Apr  7 03:15 ..
-rw-r--r-- 1 root root  3303 Apr  7 03:15 config.py
-rw-r--r-- 1 root root 25804 Apr  7 03:15 dataset.py
-rw-r--r-- 1 root root 30968 Apr  7 03:15 model.py
-rw-r--r-- 1 root root  1066 Apr  7 03:15 scheduler.py
-rw-r--r-- 1 root root 24150 Apr  7 03:15 train.py
-rw-r--r-- 1 root root  9482 Apr  7 03:15 utils.py
-rw-r--r-- 1 root root  3113 Apr  7 03:15 vocabulary.py



## 5️⃣ Kaggle API Setup
Vào **Edit → Secrets (🔒)** → thêm `KAGGLE_USERNAME` và `KAGGLE_KEY`


In [2]:
import os

# Cách 1: Colab Secrets (khuyến nghị)
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    print('✓ Kaggle credentials loaded from Colab Secrets')
except Exception as e:
    print(f'Secrets không có ({e}) → dùng kaggle.json...')
    from google.colab import files
    uploaded = files.upload()  # upload kaggle.json
    import shutil
    os.makedirs(os.path.expanduser('~/.config/kaggle'), exist_ok=True)
    shutil.move('kaggle.json', os.path.expanduser('~/.config/kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.config/kaggle/kaggle.json'), 0o600)
    print('✓ kaggle.json installed')

# Test
import subprocess
r = subprocess.run(['python', '-c', 'import kagglehub; print("kagglehub:", kagglehub.__version__)'],
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())


✓ Kaggle credentials loaded from Colab Secrets
kagglehub: 1.0.0


## 6️⃣ Download WLASL Dataset
> Dataset `risangbaskoro/wlasl-processed` (~4.8GB, 11,980 videos, 2000 classes)  
> Nếu đã download rồi (Drive có sẵn) → chạy cell dưới để restore, bỏ qua cell này


In [4]:
import kagglehub, os, shutil, glob

# Khôi phục lại các biến đường dẫn nếu bị mất sau khi Restart Runtime
DRIVE_ROOT = '/content/drive/MyDrive/sign_language_wlasl'
DRIVE_DATA = f'{DRIVE_ROOT}/data'

WLASL_LOCAL = '/content/data/wlasl_raw'
os.makedirs(WLASL_LOCAL, exist_ok=True)

# Kiểm tra xem Drive đã có data chưa
drive_wlasl = f'{DRIVE_DATA}/wlasl_raw'
if os.path.exists(f'{drive_wlasl}/videos') and len(glob.glob(f'{drive_wlasl}/videos/*.mp4')) > 100:
    print(f'✓ Found WLASL on Drive ({len(glob.glob(f"{drive_wlasl}/videos/*.mp4"))} videos)')
    print('  Symlinking Drive → /content/data/wlasl_raw ...')
    if not os.path.exists(WLASL_LOCAL):
        os.symlink(drive_wlasl, WLASL_LOCAL)
    else:
        # Copy JSON annotation files saja (nhanh)
        for f in glob.glob(f'{drive_wlasl}/*.json') + glob.glob(f'{drive_wlasl}/*.txt'):
            shutil.copy(f, WLASL_LOCAL)
        # Symlink videos folder
        video_local = f'{WLASL_LOCAL}/videos'
        if not os.path.exists(video_local):
            os.symlink(f'{drive_wlasl}/videos', video_local)
    print('✓ Done — tiếp tục Cell 7')
else:
    print('📦 Downloading WLASL from Kaggle (~4.8GB, ~5-10 phút)...')
    wlasl_path = kagglehub.dataset_download(
        'risangbaskoro/wlasl-processed',
        output_dir=WLASL_LOCAL
    )
    print(f'✓ Downloaded to: {wlasl_path}')

    # Backup JSON files lên Drive (nhẹ, không backup video vì 4.8GB)
    os.makedirs(drive_wlasl, exist_ok=True)
    for f in glob.glob(f'{WLASL_LOCAL}/*.json') + glob.glob(f'{WLASL_LOCAL}/*.txt'):
        shutil.copy(f, drive_wlasl)
        print(f'  Backed up: {os.path.basename(f)}')

# Verify
mp4s  = glob.glob(f'{WLASL_LOCAL}/videos/*.mp4')
jsons = glob.glob(f'{WLASL_LOCAL}/*.json')
print(f'\n✓ Videos  : {len(mp4s)}')
print(f'✓ JSONs   : {[os.path.basename(j) for j in jsons]}')

📦 Downloading WLASL from Kaggle (~4.8GB, ~5-10 phút)...


100%|██████████| 4.82G/4.82G [05:31<00:00, 15.6MB/s]

Extracting files...


✓ Downloaded to: /content/data/wlasl_raw
  Backed up: nslt_2000.json
  Backed up: nslt_300.json
  Backed up: nslt_1000.json
  Backed up: WLASL_v0.3.json
  Backed up: nslt_100.json
  Backed up: missing.txt
  Backed up: wlasl_class_list.txt

✓ Videos  : 11980
✓ JSONs   : ['nslt_2000.json', 'nslt_300.json', 'nslt_1000.json', 'WLASL_v0.3.json', 'nslt_100.json']


## 7️⃣ Extract Keypoints từ Video
> Dùng `nslt_100.json` (100 class, ~2,000 videos) để train **nhanh và tốt hơn** trên Colab T4.  
> `nslt_100.json` có train/val/test split sẵn, cân bằng dữ liệu hơn `nslt_2000.json`.  
> **Thời gian ước tính:** ~30-60 phút trên T4 (có resume nếu bị ngắt)


In [5]:
import json, os, glob, cv2, numpy as np, mediapipe as mp
from pathlib import Path
from tqdm.notebook import tqdm
import shutil

# ── Config ──
WLASL_LOCAL   = '/content/data/wlasl_raw'
KP_DIR        = '/content/data/keypoints'
DATA_DIR      = '/content/data'
ANNOT_FILE    = f'{WLASL_LOCAL}/nslt_100.json'   # 100 class (~2000 videos)
MAX_FRAMES    = 128    # T4: giới hạn để tiết kiệm RAM
TARGET_FPS    = 25
os.makedirs(KP_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# ── Load annotation ──
# nslt_100.json format: {video_id: {subset: 'train'/'val'/'test', action: [class_id, start, end]}}
with open(ANNOT_FILE) as f:
    all_annot = json.load(f)
print(f'✓ Annotation: {len(all_annot)} video entries, 100 classes')

# ── Load class list ──
class_list_file = f'{WLASL_LOCAL}/wlasl_class_list.txt'
with open(class_list_file) as f:
    class_lines = [l.strip().split('\t') for l in f if l.strip()]
# format: index\tword
id_to_word = {int(idx): word for idx, word in class_lines}
print(f'✓ Class list: {len(id_to_word)} classes')
print(f'  Sample: {dict(list(id_to_word.items())[:5])}')

# ── Map video files ──
all_videos = glob.glob(f'{WLASL_LOCAL}/videos/*.mp4')
video_map  = {Path(v).stem: v for v in all_videos}
print(f'✓ Videos on disk: {len(video_map)}')

# ── Khởi tạo MediaPipe (Solutions API — nhất quán với dataset.py) ──
mp_holistic = mp.solutions.holistic

def extract_video_keypoints(video_path, max_frames=MAX_FRAMES, target_fps=TARGET_FPS):
    """
    Extract (T, 543, 4) keypoints từ video.
    Dùng mp.solutions.holistic — nhất quán với dataset.py.
    """
    kp_list = []
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    orig_fps   = cap.get(cv2.CAP_PROP_FPS) or 25.0
    frame_skip = max(1, round(orig_fps / target_fps))

    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=0,          # 0=lite, nhanh cho batch extract
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as holistic:
        frame_idx = 0
        while cap.isOpened() and len(kp_list) < max_frames:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % frame_skip == 0:
                kp = np.zeros((543, 4), dtype=np.float32)
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                res = holistic.process(rgb)

                def fill(landmarks, start, n):
                    if landmarks:
                        for i, lm in enumerate(landmarks.landmark[:n]):
                            kp[start + i] = [
                                lm.x, lm.y, lm.z,
                                lm.visibility if hasattr(lm, 'visibility') else 1.0
                            ]

                fill(res.pose_landmarks,       0,  33)
                fill(res.left_hand_landmarks,  33, 21)
                fill(res.right_hand_landmarks, 54, 21)
                fill(res.face_landmarks,       75, 468)
                kp_list.append(kp)
            frame_idx += 1

    cap.release()
    if len(kp_list) < 2:
        return None
    return np.stack(kp_list, axis=0).astype(np.float32)   # (T, 543, 4)

# ── Extract keypoints (có resume nếu bị ngắt) ──
samples  = []
failed   = []
skipped  = 0

items = [(vid_id, info) for vid_id, info in all_annot.items() if vid_id in video_map]
print(f'\nProcessing {len(items)} videos (skipping {len(all_annot)-len(items)} not on disk)...')

for vid_id, info in tqdm(items, desc='Extracting'):
    out_path  = f'{KP_DIR}/{vid_id}.npy'
    class_id  = info['action'][0]
    word      = id_to_word.get(class_id, str(class_id))
    subset    = info.get('subset', 'train')

    sample = {
        'id'            : vid_id,
        'keypoints_path': out_path,
        'gloss'         : word,
        'text'          : word,
        'dataset'       : 'wlasl',
        'split'         : subset,
        'class_id'      : class_id,
    }

    # Resume: skip nếu đã extract
    if os.path.exists(out_path):
        try:
            shape = np.load(out_path, mmap_mode='r').shape
            if shape[0] >= 2 and shape[1] == 543 and shape[2] == 4:
                samples.append(sample)
                skipped += 1
                continue
        except Exception:
            pass  # file corrupt → re-extract

    kps = extract_video_keypoints(video_map[vid_id])
    if kps is None:
        failed.append(vid_id)
        continue

    np.save(out_path, kps)
    samples.append(sample)

print(f'\n✓ Extracted : {len(samples) - skipped} new')
print(f'✓ Resumed   : {skipped} skipped (already done)')
print(f'✗ Failed    : {len(failed)}')
print(f'→ Total samples: {len(samples)}')

# ── Backup keypoints lên Drive ──
drive_kp = f'{DRIVE_DATA}/keypoints'
if not os.path.exists(drive_kp):
    print(f'\nBacking up keypoints to Drive...')
    shutil.copytree(KP_DIR, drive_kp)
    print(f'✓ Backed up {len(os.listdir(drive_kp))} files to {drive_kp}')
else:
    # Chỉ copy file mới
    existing = set(os.listdir(drive_kp))
    new_files = [f for f in os.listdir(KP_DIR) if f not in existing]
    for f in new_files:
        shutil.copy(f'{KP_DIR}/{f}', f'{drive_kp}/{f}')
    print(f'✓ Backed up {len(new_files)} new keypoint files to Drive')

# ── Split và lưu JSON ──
# Giữ nguyên subset từ annotation (train/val/test đã được WLASL định nghĩa)
train_s = [s for s in samples if s['split'] == 'train']
val_s   = [s for s in samples if s['split'] == 'val']
test_s  = [s for s in samples if s['split'] == 'test']

# Nếu không có split info → tự chia 80/10/10
if len(val_s) == 0:
    import random; random.seed(42); random.shuffle(samples)
    n = len(samples)
    train_s = samples[:int(n*0.8)]
    val_s   = samples[int(n*0.8):int(n*0.9)]
    test_s  = samples[int(n*0.9):]
    for s in train_s: s['split'] = 'train'
    for s in val_s:   s['split'] = 'val'
    for s in test_s:  s['split'] = 'test'

with open(f'{DATA_DIR}/train.json', 'w') as f: json.dump(train_s, f)
with open(f'{DATA_DIR}/val.json',   'w') as f: json.dump(val_s,   f)
with open(f'{DATA_DIR}/test.json',  'w') as f: json.dump(test_s,  f)

# Backup JSON lên Drive
for split in ['train', 'val', 'test']:
    shutil.copy(f'{DATA_DIR}/{split}.json', f'{DRIVE_DATA}/{split}.json')

print(f'\n✓ Train : {len(train_s)} samples')
print(f'✓ Val   : {len(val_s)} samples')
print(f'✓ Test  : {len(test_s)} samples')
print(f'✓ Saved to /content/data/ and Drive')


✓ Annotation: 2038 video entries, 100 classes
✓ Class list: 2000 classes
  Sample: {0: 'book', 1: 'drink', 2: 'computer', 3: 'before', 4: 'chair'}
✓ Videos on disk: 11980

Processing 1013 videos (skipping 1025 not on disk)...


Extracting:   0%|          | 0/1013 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '



✓ Extracted : 1013 new
✓ Resumed   : 0 skipped (already done)
✗ Failed    : 0
→ Total samples: 1013

Backing up keypoints to Drive...
✓ Backed up 1013 files to /content/drive/MyDrive/sign_language_wlasl/data/keypoints

✓ Train : 748 samples
✓ Val   : 165 samples
✓ Test  : 100 samples
✓ Saved to /content/data/ and Drive


## 8️⃣ Build Vocabulary + Config

In [7]:
import json, os, sys, yaml, torch
sys.path.insert(0, '/content/sign_language')

from vocabulary import Vocabulary

# ── Khôi phục cấu hình phần cứng nếu bị mất sau Restart Runtime ──
if 'D_MODEL' not in globals() or 'BATCH_SIZE' not in globals():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb < 14: BATCH_SIZE = 2; D_MODEL = 128
    elif vram_gb < 20: BATCH_SIZE = 4; D_MODEL = 256
    else: BATCH_SIZE = 8; D_MODEL = 256
    print(f'ℹ️ Re-initialized config: BATCH_SIZE={BATCH_SIZE}, D_MODEL={D_MODEL}')

DRIVE_ROOT = '/content/drive/MyDrive/sign_language_wlasl'
DRIVE_DATA = f'{DRIVE_ROOT}/data'
DATA_DIR  = '/content/data'

# ── Build Vocabulary ──
with open(f'{DATA_DIR}/train.json') as f:
    train_samples = json.load(f)

gloss_vocab = Vocabulary.build_from_json(f'{DATA_DIR}/train.json', field='gloss')
gloss_vocab.save(f'{DATA_DIR}/gloss_vocab.json')
print(f'✓ Gloss vocab: {len(gloss_vocab)} tokens')

text_vocab = Vocabulary.build_from_json(f'{DATA_DIR}/train.json', field='text')
text_vocab.save(f'{DATA_DIR}/text_vocab.json')
print(f'✓ Text  vocab: {len(text_vocab)} tokens')

# Backup vocab lên Drive
import shutil
for f in ['gloss_vocab.json', 'text_vocab.json']:
    shutil.copy(f'{DATA_DIR}/{f}', f'{DRIVE_DATA}/{f}')
print(f'✓ Vocab backed up to Drive')

# ── Write Config ──
os.makedirs('/content/sign_language/configs', exist_ok=True)

num_classes = len(gloss_vocab)
print(f'\nNum classes (gloss): {num_classes}')

config_colab = {
    'train_data_path' : '/content/data/train.json',
    'val_data_path'   : '/content/data/val.json',
    'test_data_path'  : '/content/data/test.json',
    'gloss_vocab_path': '/content/data/gloss_vocab.json',
    'text_vocab_path' : '/content/data/text_vocab.json',
    'num_keypoints'   : 543,
    'keypoint_dim'    : 4,
    'max_seq_len'     : 128,
    'max_text_len'    : 16,
    'num_workers'     : 2,
    'dataset_mode'    : 'isolated',
    'd_model'                : D_MODEL,
    'num_graph_layers'       : 2,
    'num_temporal_layers'    : 3,
    'num_heads'              : 8,
    'temporal_window_size'   : 5,
    'dropout'                : 0.1,
    'decoder_name'           : 'facebook/mbart-large-50',
    'gradient_checkpointing' : True,
    'epochs'                     : 30,
    'batch_size'                 : BATCH_SIZE,
    'eval_batch_size'            : BATCH_SIZE * 2,
    'gradient_accumulation_steps': 8,
    'lr'                         : 1e-3,
    'pretrained_lr_scale'        : 0.01,
    'weight_decay'               : 1e-4,
    'max_grad_norm'              : 1.0,
    'warmup_ratio'               : 0.1,
    'patience'                   : 10,
    'use_amp'                    : True,
    'seed'                       : 42,
    'lambda_ctc' : 0.0,
    'lambda_ce'  : 1.0,
    'aug_flip_prob'    : 0.5,
    'aug_noise_std'    : 0.01,
    'aug_scale_range'  : [0.85, 1.15],
    'aug_rotation'     : 15.0,
    'aug_time_stretch' : [0.8, 1.2],
    'aug_frame_drop'   : 0.1,
    'aug_joint_drop'   : 0.05,
    'log_interval': 20,
    'output_dir'  : '/content/outputs/',
    'drive_backup': f'{DRIVE_ROOT}/checkpoints',
}

config_path = '/content/sign_language/configs/colab.yaml'
with open(config_path, 'w') as f:
    yaml.dump(config_colab, f)

print('\n✓ Config saved to:', config_path)
print('\nKey settings:')
for k in ['dataset_mode', 'lambda_ctc', 'lambda_ce', 'd_model', 'batch_size', 'epochs', 'lr', 'max_seq_len']:
    print(f'  {k}: {config_colab[k]}')

ℹ️ Re-initialized config: BATCH_SIZE=4, D_MODEL=256
✓ Gloss vocab: 104 tokens
✓ Text  vocab: 104 tokens
✓ Vocab backed up to Drive

Num classes (gloss): 104

✓ Config saved to: /content/sign_language/configs/colab.yaml

Key settings:
  dataset_mode: isolated
  lambda_ctc: 0.0
  lambda_ce: 1.0
  d_model: 256
  batch_size: 4
  epochs: 30
  lr: 0.001
  max_seq_len: 128


## 9️⃣ Sanity Check — Forward Pass

In [9]:
import sys, torch, os
sys.path.insert(0, '/content/sign_language')

from config import get_config
from model import UpgradedHSTGNN
from vocabulary import Vocabulary

config      = get_config('/content/sign_language/configs/colab.yaml')
gloss_vocab = Vocabulary.load('/content/data/gloss_vocab.json')
text_vocab  = Vocabulary.load('/content/data/text_vocab.json')

print(f'Device      : {config.device}')
print(f'Gloss vocab : {len(gloss_vocab)} tokens')
print(f'Text vocab  : {len(text_vocab)} tokens')
print(f'Dataset mode: {getattr(config, "dataset_mode", "continuous")}')
print(f'Lambda CTC  : {config.lambda_ctc}  ← phải = 0.0 cho WLASL')
print(f'Lambda CE   : {config.lambda_ce}   ← phải = 1.0 cho WLASL')

model = UpgradedHSTGNN(
    num_keypoints            = config.num_keypoints,
    keypoint_dim             = config.keypoint_dim,
    d_model                  = config.d_model,
    num_graph_layers         = config.num_graph_layers,
    num_heads                = config.num_heads,
    num_gloss_classes        = len(gloss_vocab),
    text_vocab_size          = len(text_vocab),
    temporal_window_size     = getattr(config, 'temporal_window_size', 5),
    decoder_name             = config.decoder_name,
    dropout                  = config.dropout,
    use_gradient_checkpointing = config.gradient_checkpointing,
).to(config.device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal params    : {total/1e6:.2f}M')
print(f'Trainable params: {trainable/1e6:.2f}M')
print(f'Frozen params   : {(total-trainable)/1e6:.2f}M (mBART lower layers)')

# Forward pass test
B, T, N, C = 2, 32, 543, 4
kps     = torch.randn(B, T, N, C, device=config.device)
kp_lens = torch.tensor([32, 28], device=config.device)
gloss_t = torch.randint(1, max(len(gloss_vocab), 2), (B, 3), device=config.device)
text_t  = torch.randint(1, max(len(text_vocab), 2),  (B, 4), device=config.device)

try:
    with torch.cuda.amp.autocast():
        out = model(kps, kp_lens, gloss_t, text_t)
    print('\n✓ Forward pass OK!')
    for k, v in out.items():
        if hasattr(v, 'shape'):
            print(f'  {k}: {v.shape}')
except Exception as e:
    print(f'\n❌ Forward pass FAILED: {e}')
    raise

vram = torch.cuda.memory_allocated() / 1e9
print(f'\nVRAM used: {vram:.2f} GB  (limit: 15 GB)')
if vram > 12:
    print('⚠️  VRAM cao — có thể OOM khi train với batch đầy. Giảm batch_size hoặc max_seq_len.')
else:
    print('✓ VRAM OK')


Device      : cuda
Gloss vocab : 104 tokens
Text vocab  : 104 tokens
Dataset mode: continuous
Lambda CTC  : 0.0  ← phải = 0.0 cho WLASL
Lambda CE   : 1.0   ← phải = 1.0 cho WLASL


/content/sign_language/model.py:402: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]


Total params    : 620.20M
Trainable params: 227.66M
Frozen params   : 392.53M (mBART lower layers)


/tmp/ipykernel_2273/2114089642.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



✓ Forward pass OK!
  gloss_log_probs: torch.Size([16, 2, 104])
  text_logits: torch.Size([2, 4, 250054])
  encoder_hidden: torch.Size([2, 16, 256])
  encoder_lengths: torch.Size([2])

VRAM used: 3.47 GB  (limit: 15 GB)
✓ VRAM OK


## 🔄 Auto-Resume Logic
Mỗi session mới chạy cell này để tự động restore checkpoint từ Drive.


In [10]:
import os, shutil

LOCAL_OUTPUT = '/content/outputs'
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

local_latest = f'{LOCAL_OUTPUT}/checkpoint_latest.pt'
drive_latest = f'{DRIVE_CKPT}/checkpoint_latest.pt'
RESUME_PATH  = None

if os.path.exists(local_latest):
    RESUME_PATH = local_latest
    size = os.path.getsize(local_latest) / 1e6
    print(f'✓ Local checkpoint found: {size:.1f} MB')

elif os.path.exists(drive_latest):
    print('Restoring checkpoint from Drive...')
    shutil.copy(drive_latest, local_latest)
    RESUME_PATH = local_latest
    size = os.path.getsize(local_latest) / 1e6
    print(f'✓ Restored from Drive: {size:.1f} MB')

    drive_best = f'{DRIVE_CKPT}/checkpoint_best.pt'
    if os.path.exists(drive_best):
        shutil.copy(drive_best, f'{LOCAL_OUTPUT}/checkpoint_best.pt')
        print('✓ Best checkpoint also restored')
else:
    print('No checkpoint found → Fresh training')

print(f'\nResume path: {RESUME_PATH}')

# Hiện danh sách checkpoints trên Drive
if os.path.exists(DRIVE_CKPT):
    ckpts = sorted([f for f in os.listdir(DRIVE_CKPT) if f.endswith('.pt')])
    if ckpts:
        print(f'\nCheckpoints on Drive:')
        for c in ckpts:
            size = os.path.getsize(f'{DRIVE_CKPT}/{c}') / 1e6
            print(f'  {c}: {size:.1f} MB')


No checkpoint found → Fresh training

Resume path: None


## 🚀 Train!
### Chiến lược 3 session (Colab free T4):

| Session | Phase | Mô tả | Mục tiêu |
|---------|-------|-------|----------|
| 1 | `phase=1` | Pretrain Encoder (freeze mBART) | Acc > 20% |
| 2 | `phase=2` | Fine-tune Decoder (freeze Encoder) | Acc > 35% |
| 3+ | `phase=0` | Joint fine-tune | Acc > 50% |


In [11]:
import sys

# ── ĐẶT PHASE Ở ĐÂY ──
# Session 1: TRAINING_PHASE = 1
# Session 2: TRAINING_PHASE = 2
# Session 3+: TRAINING_PHASE = 0
TRAINING_PHASE = 0   # ← THAY ĐỔI THEO SESSION

cmd_parts = [
    sys.executable, '/content/sign_language/train.py',
    '--config',            '/content/sign_language/configs/colab.yaml',
    '--output_dir',        LOCAL_OUTPUT,
    '--drive_backup',      DRIVE_CKPT,
    '--phase',             str(TRAINING_PHASE),
    '--save_interval_min', '15',
]
if RESUME_PATH:
    cmd_parts += ['--resume', RESUME_PATH]

cmd = ' '.join(cmd_parts)
print('Phase:', TRAINING_PHASE, '(1=encoder, 2=decoder, 0=joint)')
print('Command:', cmd)


Phase: 0 (1=encoder, 2=decoder, 0=joint)
Command: /usr/bin/python3 /content/sign_language/train.py --config /content/sign_language/configs/colab.yaml --output_dir /content/outputs --drive_backup /content/drive/MyDrive/sign_language_wlasl/checkpoints --phase 0 --save_interval_min 15


In [3]:
# ── Chạy training ──
!{cmd}


/bin/bash: line 1: {cmd}: command not found


## 📊 Visualize Training History

In [ ]:
import json, os
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

history_path = f'{LOCAL_OUTPUT}/history.json'
if not os.path.exists(history_path):
    print('No history.json yet — train first!')
else:
    with open(history_path) as f:
        history = json.load(f)

    epochs = [h['epoch'] for h in history]
    acc    = [h.get('accuracy', 0) for h in history]      # WLASL: dùng accuracy
    loss   = [h.get('loss', 0) for h in history]
    ce_l   = [h.get('ce_loss', 0) for h in history]
    val_acc = [h.get('val_accuracy', 0) for h in history]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('HST-GNN v2 WLASL Training', fontsize=13, fontweight='bold')

    axes[0].plot(epochs, loss, 'b-o', ms=3)
    axes[0].set_title('Total Loss ↓'); axes[0].set_xlabel('Epoch'); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, ce_l, 'm-o', ms=3)
    axes[1].set_title('CE Loss ↓'); axes[1].set_xlabel('Epoch'); axes[1].grid(True, alpha=0.3)

    axes[2].plot(epochs, acc,     'g-o',  ms=3, label='Train Acc')
    axes[2].plot(epochs, val_acc, 'r--o', ms=3, label='Val Acc')
    axes[2].set_title('Accuracy ↑'); axes[2].set_xlabel('Epoch')
    axes[2].legend(); axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = f'{LOCAL_OUTPUT}/training_history.png'
    plt.savefig(save_path, bbox_inches='tight')
    import shutil
    shutil.copy(save_path, f'{DRIVE_ROOT}/training_history.png')
    plt.show()

    if val_acc:
        best_epoch = epochs[val_acc.index(max(val_acc))]
        print(f'\nBest Val Accuracy: {max(val_acc):.2f}% (epoch {best_epoch})')
    print(f'Saved: {save_path}')


## 🧪 Evaluate Best Model

In [ ]:
import os, shutil

best_ckpt = f'{LOCAL_OUTPUT}/checkpoint_best.pt'
if not os.path.exists(best_ckpt):
    drive_best = f'{DRIVE_CKPT}/checkpoint_best.pt'
    if os.path.exists(drive_best):
        shutil.copy(drive_best, best_ckpt)
        print('✓ Restored best checkpoint from Drive')
    else:
        print('❌ No best checkpoint found — train first!')

if os.path.exists(best_ckpt):
    !python /content/sign_language/train.py \
        --config /content/sign_language/configs/colab.yaml \
        --resume {best_ckpt} \
        --eval_only \
        --output_dir {LOCAL_OUTPUT}


## 🎬 Inference Demo

In [ ]:
import torch, sys, os
sys.path.insert(0, '/content/sign_language')

from config import get_config
from model import UpgradedHSTGNN
from vocabulary import Vocabulary
import numpy as np, cv2, mediapipe as mp

config      = get_config('/content/sign_language/configs/colab.yaml')
gloss_vocab = Vocabulary.load('/content/data/gloss_vocab.json')
text_vocab  = Vocabulary.load('/content/data/text_vocab.json')

model = UpgradedHSTGNN(
    num_keypoints            = config.num_keypoints,
    keypoint_dim             = config.keypoint_dim,
    d_model                  = config.d_model,
    num_graph_layers         = config.num_graph_layers,
    num_heads                = config.num_heads,
    num_gloss_classes        = len(gloss_vocab),
    text_vocab_size          = len(text_vocab),
    temporal_window_size     = getattr(config, 'temporal_window_size', 5),
    decoder_name             = config.decoder_name,
    dropout                  = config.dropout,
    use_gradient_checkpointing = False,   # inference không cần
).to(config.device)

ckpt_path = f'{LOCAL_OUTPUT}/checkpoint_best.pt'
if os.path.exists(ckpt_path):
    state = torch.load(ckpt_path, map_location=config.device)
    model.load_state_dict(state['model_state_dict'])
    model.eval()
    print('✓ Model loaded!')
else:
    print('No checkpoint found — train first!')

mp_holistic = mp.solutions.holistic

def translate_video(video_path: str) -> str:
    """Nhận video .mp4, trả về predicted sign word."""
    kp_list = []
    cap = cv2.VideoCapture(video_path)
    with mp_holistic.Holistic(
        static_image_mode=False, model_complexity=0,
        min_detection_confidence=0.5, min_tracking_confidence=0.5
    ) as holistic:
        frame_idx = 0
        while cap.isOpened() and len(kp_list) < 128:
            ret, frame = cap.read()
            if not ret: break
            if frame_idx % 1 == 0:
                kp = np.zeros((543, 4), dtype=np.float32)
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                res = holistic.process(rgb)
                def fill(lm, s, n):
                    if lm:
                        for i, l in enumerate(lm.landmark[:n]):
                            kp[s+i] = [l.x, l.y, l.z, getattr(l,'visibility',1.0)]
                fill(res.pose_landmarks, 0, 33)
                fill(res.left_hand_landmarks, 33, 21)
                fill(res.right_hand_landmarks, 54, 21)
                fill(res.face_landmarks, 75, 468)
                kp_list.append(kp)
            frame_idx += 1
    cap.release()
    if len(kp_list) < 2:
        return 'ERROR: video too short'

    kps_t   = torch.from_numpy(np.stack(kp_list)).unsqueeze(0).to(config.device)
    kp_lens = torch.tensor([len(kp_list)], device=config.device)

    with torch.no_grad():
        out   = model(kps_t, kp_lens)
        probs = out['gloss_log_probs']   # (1, T, num_classes)
        # Mean pooling over time → top-1 class
        avg   = probs.mean(dim=1)        # (1, num_classes)
        pred  = avg.argmax(dim=-1).item()

    return gloss_vocab.idx2word.get(pred, f'class_{pred}')

print('✓ translate_video() ready')
print('Usage:')
print('  from google.colab import files')
print('  uploaded = files.upload()  # upload 1 video .mp4')
print('  result = translate_video(list(uploaded.keys())[0])')
print('  print("Predicted sign:", result)')
